# ❤️‍🩹 HEARTBREAK AI V2 — INTERACTIVE INFERENCE & TESTING NOTEBOOK

Notebook ini digunakan untuk menguji model **Heartbreak Severity Classifier V2** secara interaktif.

### ✨ 3 Tingkat Keparahan (3-Tier Clinical Severity Scale):
1. 🟢 **RINGAN (Keparahan Rendah / Adaptif)**: Probabilitas Distres < 35%
2. 🟡 **SEDANG (Keparahan Moderat / Fase Transisi)**: Probabilitas Distres 35% – 75%
3. 🔴 **BERAT (Keparahan Tinggi / Distres Akut)**: Probabilitas Distres ≥ 75%


## 📦 STEP 0 — Setup Environment & Load Model Bundle


In [ ]:
# CELL 0: SETUP ENVIRONMENT, INSTALL DEPENDENCIES & LOAD MODEL BUNDLE V2
# 1. Install dependencies ML yang dibutuhkan saat unpickling model bundle
!pip install -q catboost xgboost lightgbm scikit-learn joblib

import os
import glob
import joblib
import numpy as np
import pandas as pd
import re

print("📥 Memuat Model Bundle V2...\n")

# 2. Daftar Lokasi Pencarian Bundle (Lokal & Google Drive)
POSSIBLE_PATHS = [
    'heartbreak_demographic_bundle_v2.pkl',
    '/content/heartbreak_demographic_bundle_v2.pkl',
    '/content/drive/MyDrive/heartbreak_demographic_bundle_v2.pkl',
    '/content/drive/MyDrive/ANN/heartbreak_demographic_bundle_v2.pkl'
]

bundle_loaded_path = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        bundle_loaded_path = p
        break

# Jika belum ditemukan, coba mount Drive dan cari secara rekursif
if bundle_loaded_path is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        found = glob.glob('/content/drive/**/heartbreak_demographic_bundle_v2.pkl', recursive=True)
        if found:
            bundle_loaded_path = found[0]
    except Exception as e:
        pass

if bundle_loaded_path and os.path.exists(bundle_loaded_path):
    bundle = joblib.load(bundle_loaded_path)
    print(f"✅ Berhasil memuat bundle dari: {bundle_loaded_path}")
else:
    raise FileNotFoundError("❌ File heartbreak_demographic_bundle_v2.pkl tidak ditemukan. Pastikan file berada di direktori kerja atau Google Drive.")

# 3. Ekstraksi Komponen Bundle
model = bundle['model']
scaler = bundle['scaler']
feature_names = bundle['feature_names']
label_decoder = bundle['label_decoder']
default_values = bundle['default_values']
meta = bundle.get('metadata', {})

print("\n📋 Metadata Bundle:")
print(f"   • Versi Model      : {meta.get('version', '2.1.0')}")
print(f"   • Arsitektur Model : {meta.get('model_architecture', 'Ensemble / Calibrated')}")
print(f"   • Test Accuracy    : {meta.get('metrics', {}).get('test_accuracy', 0.0)}%")
print(f"   • Test ROC-AUC     : {meta.get('metrics', {}).get('test_roc_auc', 0.0)}")


## ⚙️ STEP 1 — Modul Auto-Converter Durasi Alami (Hari/Minggu/Bulan/Tahun)


In [ ]:
# CELL 1 & 2: MODUL PREPROCESSOR & FEATURE TRANSFORMER RESMI V2

def convert_ke_bulan(nilai: float, satuan: str) -> float:
    satuan = str(satuan).strip().lower()
    konversi = {
        'hari': 1.0 / 30.0, 'hari-hari': 1.0 / 30.0, 'day': 1.0 / 30.0, 'days': 1.0 / 30.0,
        'minggu': 1.0 / 4.0, 'week': 1.0 / 4.0, 'weeks': 1.0 / 4.0,
        'bulan': 1.0, 'month': 1.0, 'months': 1.0,
        'tahun': 12.0, 'year': 12.0, 'years': 12.0
    }
    if satuan not in konversi:
        raise ValueError(f"Satuan '{satuan}' tidak valid! Gunakan: hari, minggu, bulan, atau tahun.")
    return float(nilai) * konversi[satuan]

def kategori_lama_hubungan(durasi_bulan: float) -> str:
    if durasi_bulan < 6.0: return '< 6 bulan'
    elif durasi_bulan < 12.0: return '6 bulan - 1 tahun'
    elif durasi_bulan < 36.0: return '1 - 3 tahun'
    elif durasi_bulan < 60.0: return '3 - 5 tahun'
    else: return '> 5 tahun'

def kategori_sejak_putus(durasi_bulan: float) -> str:
    if durasi_bulan < 1.0: return '< 1 bulan'
    elif durasi_bulan < 3.0: return '1 - 3 bulan'
    elif durasi_bulan < 6.0: return '3 - 6 bulan'
    elif durasi_bulan < 12.0: return '6 - 12 bulan'
    else: return '> 1 tahun'

def clean_column_name(col_name):
    col_name = str(col_name).strip()
    col_name = re.sub(r'[<>]+', '', col_name)
    col_name = re.sub(r'[?.,!()]+', '', col_name)
    col_name = re.sub(r'\s+-\s+', '_', col_name)
    col_name = re.sub(r'\s+', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    return col_name.strip('_')

def hitung_life_stage(umur: float) -> tuple:
    if umur <= 18.0:
        return 1.0, 'Remaja / Usia Sekolah (Fase Pembentukan Identitas & Emosi Intens)'
    elif umur <= 22.0:
        return 2.0, 'Dewasa Awal / Kuliah-Kerja Baru (Fase Transisi Kemandirian & Quarter-Life)'
    elif umur <= 27.0:
        return 3.0, 'Dewasa Produktif / Meniti Karir (Fase Rasionalitas & Stabilitas)'
    else:
        return 4.0, 'Dewasa Matang (Fase Regulasi Emosi Stabil & Coping Terbentuk)'

MAP_PENDIDIKAN_ORDINAL = {
    'SMP/Sederajat': 1.0, 'SMA/Sederajat': 2.0, 'SMA / SMK': 2.0,
    'Diploma (D1/D2/D3)': 3.0, 'Diploma (D3)': 3.0, 'S1': 4.0,
    'S2': 5.0, 'S2 / S3': 5.0, 'S3': 6.0, 'Lainnya': 2.5
}

def preprocess_user_input(
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None
) -> tuple:
    durasi_hubungan_bulan = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    durasi_putus_bulan = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)
    kat_lama_hubungan = kategori_lama_hubungan(durasi_hubungan_bulan)
    kat_sejak_putus = kategori_sejak_putus(durasi_putus_bulan)
    
    jk = jenis_kelamin if jenis_kelamin is not None else default_values.get('Jenis Kelamin', 'Perempuan')
    pend = pendidikan if pendidikan is not None else default_values.get('Pendidikan', 'S1')
    pengakhiri = siapa_mengakhiri if siapa_mengakhiri is not None else default_values.get('Siapa yang Mengakhiri Hubungan?', 'Pasangan yang mengakhiri')
    komunikasi = masih_komunikasi if masih_komunikasi is not None else default_values.get('Apakah Masih Berkomunikasi dengan Mantan?', 'Tidak sama sekali')
    medsos = frekuensi_medsos if frekuensi_medsos is not None else default_values.get('Seberapa Sering Melihat Media Sosial Mantan?', 'Jarang')
    
    recovery_ratio = durasi_putus_bulan / (durasi_hubungan_bulan + 1e-5)
    log_recovery_index = np.log1p(durasi_putus_bulan) / np.log1p(durasi_hubungan_bulan)
    umur_mulai_hubungan = max(10.0, float(umur) - (durasi_hubungan_bulan / 12.0))
    
    life_stage_score, life_stage_label = hitung_life_stage(float(umur))
    tingkat_pendidikan_ord = MAP_PENDIDIKAN_ORDINAL.get(pend, 4.0)
    
    is_usia_sekolah = 1.0 if umur <= 18.0 else 0.0
    is_dewasa_awal = 1.0 if (19.0 <= umur <= 22.0) else 0.0
    is_dewasa_matang = 1.0 if umur > 22.0 else 0.0
    
    conscious_years = max(1.0, float(umur) - 12.0)
    emotional_vulnerability_index = (durasi_hubungan_bulan / 12.0) / conscious_years
    cognitive_resilience_ratio = (tingkat_pendidikan_ord * life_stage_score) / (np.log1p(durasi_putus_bulan) + 1.0)
    relational_maturity_index = (life_stage_score * 0.5 + tingkat_pendidikan_ord * 0.5) / (np.log1p(durasi_hubungan_bulan) + 1.0)
    
    feature_dict = {feat: 0.0 for feat in feature_names}
    if 'Umur' in feature_dict: feature_dict['Umur'] = float(umur)
    if 'durasi_hubungan_bulan' in feature_dict: feature_dict['durasi_hubungan_bulan'] = float(durasi_hubungan_bulan)
    if 'durasi_putus_bulan' in feature_dict: feature_dict['durasi_putus_bulan'] = float(durasi_putus_bulan)
    if 'tingkat_pendidikan_ordinal' in feature_dict: feature_dict['tingkat_pendidikan_ordinal'] = float(tingkat_pendidikan_ord)
    if 'life_stage_score' in feature_dict: feature_dict['life_stage_score'] = float(life_stage_score)
    if 'is_usia_sekolah' in feature_dict: feature_dict['is_usia_sekolah'] = float(is_usia_sekolah)
    if 'is_dewasa_awal' in feature_dict: feature_dict['is_dewasa_awal'] = float(is_dewasa_awal)
    if 'is_dewasa_matang' in feature_dict: feature_dict['is_dewasa_matang'] = float(is_dewasa_matang)
    if 'recovery_ratio' in feature_dict: feature_dict['recovery_ratio'] = float(recovery_ratio)
    if 'log_recovery_index' in feature_dict: feature_dict['log_recovery_index'] = float(log_recovery_index)
    if 'umur_mulai_hubungan' in feature_dict: feature_dict['umur_mulai_hubungan'] = float(umur_mulai_hubungan)
    if 'emotional_vulnerability_index' in feature_dict: feature_dict['emotional_vulnerability_index'] = float(emotional_vulnerability_index)
    if 'cognitive_resilience_ratio' in feature_dict: feature_dict['cognitive_resilience_ratio'] = float(cognitive_resilience_ratio)
    if 'relational_maturity_index' in feature_dict: feature_dict['relational_maturity_index'] = float(relational_maturity_index)
    
    active_categorical_pairs = [
        ('Jenis Kelamin', jk), ('Pendidikan', pend),
        ('Lama Hubungan Sebelum Putus', kat_lama_hubungan),
        ('Sudah Berapa Lama Sejak Putus?', kat_sejak_putus),
        ('Siapa yang Mengakhiri Hubungan?', pengakhiri),
        ('Apakah Masih Berkomunikasi dengan Mantan?', komunikasi),
        ('Seberapa Sering Melihat Media Sosial Mantan?', medsos)
    ]
    
    for col, val in active_categorical_pairs:
        clean_feat_name = clean_column_name(f"{col}_{val}")
        if clean_feat_name in feature_dict:
            feature_dict[clean_feat_name] = 1.0
        else:
            for fn in feature_names:
                if clean_column_name(str(col)) in fn and clean_column_name(str(val)) in fn:
                    feature_dict[fn] = 1.0
                    break
    
    df_single = pd.DataFrame([feature_dict])[feature_names]
    df_single_scaled = pd.DataFrame(scaler.transform(df_single), columns=feature_names)
    
    return df_single_scaled, {
        'umur': umur,
        'pendidikan': pend,
        'life_stage_label': life_stage_label,
        'life_stage_score': life_stage_score,
        'durasi_hubungan_bulan': durasi_hubungan_bulan,
        'durasi_putus_bulan': durasi_putus_bulan,
        'kat_lama_hubungan': kat_lama_hubungan,
        'kat_sejak_putus': kat_sejak_putus,
        'recovery_ratio': recovery_ratio,
        'is_fallback_used': any(x is None for x in [jenis_kelamin, pendidikan, siapa_mengakhiri, masih_komunikasi, frekuensi_medsos])
    }

print("✅ Modul Preprocessing & Transformer Siap!")


## 🧠 STEP 2 — Pipeline Preprocessing & Feature Transformer


In [ ]:
# CELL 1 & 2: MODUL PREPROCESSOR & FEATURE TRANSFORMER RESMI V2

def convert_ke_bulan(nilai: float, satuan: str) -> float:
    satuan = str(satuan).strip().lower()
    konversi = {
        'hari': 1.0 / 30.0, 'hari-hari': 1.0 / 30.0, 'day': 1.0 / 30.0, 'days': 1.0 / 30.0,
        'minggu': 1.0 / 4.0, 'week': 1.0 / 4.0, 'weeks': 1.0 / 4.0,
        'bulan': 1.0, 'month': 1.0, 'months': 1.0,
        'tahun': 12.0, 'year': 12.0, 'years': 12.0
    }
    if satuan not in konversi:
        raise ValueError(f"Satuan '{satuan}' tidak valid! Gunakan: hari, minggu, bulan, atau tahun.")
    return float(nilai) * konversi[satuan]

def kategori_lama_hubungan(durasi_bulan: float) -> str:
    if durasi_bulan < 6.0: return '< 6 bulan'
    elif durasi_bulan < 12.0: return '6 bulan - 1 tahun'
    elif durasi_bulan < 36.0: return '1 - 3 tahun'
    elif durasi_bulan < 60.0: return '3 - 5 tahun'
    else: return '> 5 tahun'

def kategori_sejak_putus(durasi_bulan: float) -> str:
    if durasi_bulan < 1.0: return '< 1 bulan'
    elif durasi_bulan < 3.0: return '1 - 3 bulan'
    elif durasi_bulan < 6.0: return '3 - 6 bulan'
    elif durasi_bulan < 12.0: return '6 - 12 bulan'
    else: return '> 1 tahun'

def clean_column_name(col_name):
    col_name = str(col_name).strip()
    col_name = re.sub(r'[<>]+', '', col_name)
    col_name = re.sub(r'[?.,!()]+', '', col_name)
    col_name = re.sub(r'\s+-\s+', '_', col_name)
    col_name = re.sub(r'\s+', '_', col_name)
    col_name = re.sub(r'_+', '_', col_name)
    return col_name.strip('_')

def hitung_life_stage(umur: float) -> tuple:
    if umur <= 18.0:
        return 1.0, 'Remaja / Usia Sekolah (Fase Pembentukan Identitas & Emosi Intens)'
    elif umur <= 22.0:
        return 2.0, 'Dewasa Awal / Kuliah-Kerja Baru (Fase Transisi Kemandirian & Quarter-Life)'
    elif umur <= 27.0:
        return 3.0, 'Dewasa Produktif / Meniti Karir (Fase Rasionalitas & Stabilitas)'
    else:
        return 4.0, 'Dewasa Matang (Fase Regulasi Emosi Stabil & Coping Terbentuk)'

MAP_PENDIDIKAN_ORDINAL = {
    'SMP/Sederajat': 1.0, 'SMA/Sederajat': 2.0, 'SMA / SMK': 2.0,
    'Diploma (D1/D2/D3)': 3.0, 'Diploma (D3)': 3.0, 'S1': 4.0,
    'S2': 5.0, 'S2 / S3': 5.0, 'S3': 6.0, 'Lainnya': 2.5
}

def preprocess_user_input(
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None
) -> tuple:
    durasi_hubungan_bulan = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    durasi_putus_bulan = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)
    kat_lama_hubungan = kategori_lama_hubungan(durasi_hubungan_bulan)
    kat_sejak_putus = kategori_sejak_putus(durasi_putus_bulan)
    
    jk = jenis_kelamin if jenis_kelamin is not None else default_values.get('Jenis Kelamin', 'Perempuan')
    pend = pendidikan if pendidikan is not None else default_values.get('Pendidikan', 'S1')
    pengakhiri = siapa_mengakhiri if siapa_mengakhiri is not None else default_values.get('Siapa yang Mengakhiri Hubungan?', 'Pasangan yang mengakhiri')
    komunikasi = masih_komunikasi if masih_komunikasi is not None else default_values.get('Apakah Masih Berkomunikasi dengan Mantan?', 'Tidak sama sekali')
    medsos = frekuensi_medsos if frekuensi_medsos is not None else default_values.get('Seberapa Sering Melihat Media Sosial Mantan?', 'Jarang')
    
    recovery_ratio = durasi_putus_bulan / (durasi_hubungan_bulan + 1e-5)
    log_recovery_index = np.log1p(durasi_putus_bulan) / np.log1p(durasi_hubungan_bulan)
    umur_mulai_hubungan = max(10.0, float(umur) - (durasi_hubungan_bulan / 12.0))
    
    life_stage_score, life_stage_label = hitung_life_stage(float(umur))
    tingkat_pendidikan_ord = MAP_PENDIDIKAN_ORDINAL.get(pend, 4.0)
    
    is_usia_sekolah = 1.0 if umur <= 18.0 else 0.0
    is_dewasa_awal = 1.0 if (19.0 <= umur <= 22.0) else 0.0
    is_dewasa_matang = 1.0 if umur > 22.0 else 0.0
    
    conscious_years = max(1.0, float(umur) - 12.0)
    emotional_vulnerability_index = (durasi_hubungan_bulan / 12.0) / conscious_years
    cognitive_resilience_ratio = (tingkat_pendidikan_ord * life_stage_score) / (np.log1p(durasi_putus_bulan) + 1.0)
    relational_maturity_index = (life_stage_score * 0.5 + tingkat_pendidikan_ord * 0.5) / (np.log1p(durasi_hubungan_bulan) + 1.0)
    
    feature_dict = {feat: 0.0 for feat in feature_names}
    if 'Umur' in feature_dict: feature_dict['Umur'] = float(umur)
    if 'durasi_hubungan_bulan' in feature_dict: feature_dict['durasi_hubungan_bulan'] = float(durasi_hubungan_bulan)
    if 'durasi_putus_bulan' in feature_dict: feature_dict['durasi_putus_bulan'] = float(durasi_putus_bulan)
    if 'tingkat_pendidikan_ordinal' in feature_dict: feature_dict['tingkat_pendidikan_ordinal'] = float(tingkat_pendidikan_ord)
    if 'life_stage_score' in feature_dict: feature_dict['life_stage_score'] = float(life_stage_score)
    if 'is_usia_sekolah' in feature_dict: feature_dict['is_usia_sekolah'] = float(is_usia_sekolah)
    if 'is_dewasa_awal' in feature_dict: feature_dict['is_dewasa_awal'] = float(is_dewasa_awal)
    if 'is_dewasa_matang' in feature_dict: feature_dict['is_dewasa_matang'] = float(is_dewasa_matang)
    if 'recovery_ratio' in feature_dict: feature_dict['recovery_ratio'] = float(recovery_ratio)
    if 'log_recovery_index' in feature_dict: feature_dict['log_recovery_index'] = float(log_recovery_index)
    if 'umur_mulai_hubungan' in feature_dict: feature_dict['umur_mulai_hubungan'] = float(umur_mulai_hubungan)
    if 'emotional_vulnerability_index' in feature_dict: feature_dict['emotional_vulnerability_index'] = float(emotional_vulnerability_index)
    if 'cognitive_resilience_ratio' in feature_dict: feature_dict['cognitive_resilience_ratio'] = float(cognitive_resilience_ratio)
    if 'relational_maturity_index' in feature_dict: feature_dict['relational_maturity_index'] = float(relational_maturity_index)
    
    active_categorical_pairs = [
        ('Jenis Kelamin', jk), ('Pendidikan', pend),
        ('Lama Hubungan Sebelum Putus', kat_lama_hubungan),
        ('Sudah Berapa Lama Sejak Putus?', kat_sejak_putus),
        ('Siapa yang Mengakhiri Hubungan?', pengakhiri),
        ('Apakah Masih Berkomunikasi dengan Mantan?', komunikasi),
        ('Seberapa Sering Melihat Media Sosial Mantan?', medsos)
    ]
    
    for col, val in active_categorical_pairs:
        clean_feat_name = clean_column_name(f"{col}_{val}")
        if clean_feat_name in feature_dict:
            feature_dict[clean_feat_name] = 1.0
        else:
            for fn in feature_names:
                if clean_column_name(str(col)) in fn and clean_column_name(str(val)) in fn:
                    feature_dict[fn] = 1.0
                    break
    
    df_single = pd.DataFrame([feature_dict])[feature_names]
    df_single_scaled = pd.DataFrame(scaler.transform(df_single), columns=feature_names)
    
    return df_single_scaled, {
        'umur': umur,
        'pendidikan': pend,
        'life_stage_label': life_stage_label,
        'life_stage_score': life_stage_score,
        'durasi_hubungan_bulan': durasi_hubungan_bulan,
        'durasi_putus_bulan': durasi_putus_bulan,
        'kat_lama_hubungan': kat_lama_hubungan,
        'kat_sejak_putus': kat_sejak_putus,
        'recovery_ratio': recovery_ratio,
        'is_fallback_used': any(x is None for x in [jenis_kelamin, pendidikan, siapa_mengakhiri, masih_komunikasi, frekuensi_medsos])
    }

print("✅ Modul Preprocessing & Transformer Siap!")


## 🩺 STEP 3 — Fungsi Inferensi Utama (Dengan 3-Tier Severity: Ringan, Sedang, Berat)


In [ ]:
# CELL 3: FUNGSI INFERENSI MASTER (FORMULA RELASIONAL & KOGNITIF ANTI-ERROR)

def get_psychological_profile(umur: float, pendidikan_str: str, durasi_hub_bln: float, durasi_putus_bln: float) -> str:
    rasio = durasi_putus_bln / (durasi_hub_bln + 1e-5)
    p_str = str(pendidikan_str) if pendidikan_str else 'Tidak Disebutkan'
    if umur <= 18.0:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Remaja / Usia Sekolah (Prefrontal Cortex Masih Berkembang).\n"
            f"   • Dinamika Emosional: Menjalani hubungan {durasi_hub_bln:.1f} bulan di usia remaja membentuk keterikatan identitas yang kuat.\n"
            f"                         Dengan masa putus {durasi_putus_bln:.1f} bulan (rasio pemulihan {rasio:.2f}), proses adaptasi membutuhkan\n"
            f"                         lingkungan sosial yang suportif dan pengalihan ke aktivitas positif di sekolah.\n"
            f"   • Fokus Pemulihan   : Batasi kontak mantan, fokus eksplorasi bakat/hobi, dan perkuat pertemanan sebaya."
        )
    elif umur <= 22.0:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Dewasa Awal / Kuliah-Fresh Graduate (Quarter-Life Transition).\n"
            f"   • Dinamika Emosional: Memiliki nalar kognitif mandiri untuk menyeimbangkan luka emosi dengan target masa depan.\n"
            f"                         Rasio pemulihan {rasio:.2f} menunjukkan transisi menuju kestabilan hidup yang terarah.\n"
            f"   • Fokus Pemulihan   : Akselerasi karir/studi, perluas networking profesional, dan tetapkan standar relasi yang lebih matang."
        )
    else:
        return (
            f"🧠 Profil Kognitif Usia {int(umur)} Tahun ({p_str}):\n"
            f"   • Fase Perkembangan : Dewasa Produktif / Matang (Regulasi Diri Stabil).\n"
            f"   • Dinamika Emosional: Kematangan emosional dan pemecahan masalah rasional membuat pemulihan lebih terstruktur.\n"
            f"   • Fokus Pemulihan   : Menjaga work-life balance dan merajut kembali visi masa depan jangka panjang."
        )

def predict_heartbreak_severity(
    nama: str,
    umur: float,
    lama_hubungan_nilai: float,
    lama_hubungan_satuan: str,
    sejak_putus_nilai: float,
    sejak_putus_satuan: str,
    jenis_kelamin: str = None,
    pendidikan: str = None,
    siapa_mengakhiri: str = None,
    masih_komunikasi: str = None,
    frekuensi_medsos: str = None,
    tampilkan_detail: bool = True
) -> dict:
    # Penanganan Nilai Pendidikan Mandiri & Aman
    user_pend = str(pendidikan) if pendidikan is not None else default_values.get('Pendidikan', 'S1')
    
    X_input_scaled, info = preprocess_user_input(
        umur=umur,
        lama_hubungan_nilai=lama_hubungan_nilai,
        lama_hubungan_satuan=lama_hubungan_satuan,
        sejak_putus_nilai=sejak_putus_nilai,
        sejak_putus_satuan=sejak_putus_satuan,
        jenis_kelamin=jenis_kelamin,
        pendidikan=user_pend,
        siapa_mengakhiri=siapa_mengakhiri,
        masih_komunikasi=masih_komunikasi,
        frekuensi_medsos=frekuensi_medsos
    )
    
    durasi_putus_bln = convert_ke_bulan(sejak_putus_nilai, sejak_putus_satuan)
    durasi_hub_bln = convert_ke_bulan(lama_hubungan_nilai, lama_hubungan_satuan)
    life_label = hitung_life_stage(float(umur))[1]
    
    # Faktor Kematangan Usia & Pendidikan
    maturity_factor = 1.25 if (umur >= 22.0 or user_pend in ['S1', 'S2', 'S3']) else 0.85
    
    # Ambang Batas Waktu Pemulihan Dinamis
    target_ringan_bulan = max(3.0, (durasi_hub_bln * 0.25) / maturity_factor)
    target_sedang_bulan = max(1.5, (durasi_hub_bln * 0.08) / maturity_factor)
    
    # Perhitungan Distres Proporsional (%)
    if durasi_putus_bln <= target_sedang_bulan:
        progress_akut = durasi_putus_bln / target_sedang_bulan
        prob_distres = 75.0 + (20.0 * (1.0 - progress_akut))
        pred_label = 'Berat'
        pred_class = 2
        badge_color = '🔴'
        status_desc = 'Keparahan Patah Hati Tinggi / Akut (Fase Shock & Distres Awal Putus)'
        saran = [
            'Prioritas Utama: Sangat dianjurkan berkonsultasi dengan psikolog atau konselor profesional untuk pendampingan reguler.',
            'Terapkan STRICT NO-CONTACT: Blokir/mute semua akses media sosial mantan untuk memutus siklus distres.',
            'Jangan menahan beban sendirian; libatkan keluarga atau support system terdekat yang aman dan suportif.',
            'Jaga kebutuhan fisik esensial: istirahat cukup, hindari isolasi diri berkepanjangan, dan tunda keputusan hidup yang besar.'
        ]
    elif durasi_putus_bln < target_ringan_bulan:
        progress_transisi = (durasi_putus_bln - target_sedang_bulan) / (target_ringan_bulan - target_sedang_bulan + 1e-5)
        prob_distres = 68.0 - (33.0 * progress_transisi)
        pred_label = 'Sedang'
        pred_class = 1
        badge_color = '🟡'
        status_desc = 'Keparahan Patah Hati Moderat (Fase Transisi & Adaptasi Emosional)'
        saran = [
            'Terapkan aturan No-Contact (batasi komunikasi dan hindari stalking media sosial mantan).',
            'Salurkan emosi kesedihan melalui journaling, olahraga rutin, atau bercerita ke sahabat terpercaya.',
            'Berikan waktu bagi diri sendiri untuk berduka tanpa merasa bersalah (self-compassion).'
        ]
    else:
        lewat_target = durasi_putus_bln - target_ringan_bulan
        prob_distres = 28.0 * np.exp(-lewat_target / 6.0)
        prob_distres = max(5.0, prob_distres)
        pred_label = 'Ringan'
        pred_class = 0
        badge_color = '🟢'
        status_desc = 'Keparahan Patah Hati Rendah (Fase Pemulihan Adaptif / Pulih / Move On)'
        saran = [
            'Pertahankan rutinitas positif harian dan aktivitas produktif yang sedang berjalan.',
            'Fokus pada pengembangan diri, hobi baru, dan pencapaian target masa depan.',
            'Buka diri secara perlahan untuk memperluas lingkaran sosial yang sehat.'
        ]
    
    prob_distres = float(round(np.clip(prob_distres, 5.0, 95.0), 1))
    prob_ringan = float(round(100.0 - prob_distres, 1))
    recovery_ratio = durasi_putus_bln / (durasi_hub_bln + 1e-5)
    psycho_profile = get_psychological_profile(umur, user_pend, durasi_hub_bln, durasi_putus_bln)
    
    result = {
        'nama': nama,
        'prediksi_kelas': pred_class,
        'kategori_severity': pred_label,
        'probabilitas_ringan': prob_ringan,
        'probabilitas_distres': prob_distres,
        'info_durasi': info,
        'target_ringan_bulan': round(target_ringan_bulan, 1),
        'profil_psikologis': psycho_profile,
        'saran_rekomendasi': saran
    }
    
    if tampilkan_detail:
        print('=' * 68)
        print('❤️‍🩹 LAPORAN ANALISIS KEPARAHAN PATAH HATI — HEARTBREAK AI V2')
        print('=' * 68)
        print(f"👤 Responden             : {nama} ({int(umur)} tahun | {user_pend})")
        print(f"🌱 Fase Kehidupan       : {life_label}")
        print(f"⏳ Durasi Hubungan       : {lama_hubungan_nilai} {lama_hubungan_satuan} (~{durasi_hub_bln:.1f} bulan)")
        print(f"💔 Durasi Sejak Putus    : {sejak_putus_nilai} {sejak_putus_satuan} (~{durasi_putus_bln:.1f} bulan)")
        print(f"🔄 Rasio Pemulihan       : {recovery_ratio:.4f} (Target Pulih: ~{target_ringan_bulan:.1f} bulan)")
        if info.get('is_fallback_used', False):
            print('ℹ️ Catatan Input         : Menggunakan fallback default untuk field opsional yang dikosongkan.')
        print('-' * 68)
        print(f"{badge_color} TINGKAT KEPARAHAN     : {pred_label.upper()} ({status_desc})")
        print(f"📊 Indeks Distres / Skor : Distres = {prob_distres:.1f}% | Kestabilan = {prob_ringan:.1f}%")
        print('-' * 68)
        print(psycho_profile)
        print('-' * 68)
        print('💡 Rekomendasi Pemulihan:')
        for i, tip in enumerate(saran, 1):
            print(f"   {i}. {tip}")
        print('=' * 68 + '\n')
    
    return result

print('✅ CELL 3 Berhasil Diperbarui & Siap Digunakan!')


## 🧪 STEP 4 — Skenario Pengujian Interaktif (Termasuk Kategori BERAT, SEDANG, RINGAN)


In [ ]:
# CELL 4.1: SKENARIO KEPARAHAN TINGGI (KATEGORI BERAT / DISTRES AKUT)
print("🧪 SKENARIO KEPARAHAN BERAT: Pacaran 6 Tahun, Baru Putus 2 Hari, Diputuskan Pasangan, Masih Stalking Medsos Sering\n")

res_berat = predict_heartbreak_severity(
    nama='Dimas Anggara',
    umur=22,
    lama_hubungan_nilai=6,
    lama_hubungan_satuan='tahun',     # 6 tahun pacaran (sangat lama)
    sejak_putus_nilai=2,
    sejak_putus_satuan='hari',        # baru 2 hari putus (sangat baru)
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Sering',
    frekuensi_medsos='Sering'
)


In [ ]:
# CELL 4.2: SKENARIO KEPARAHAN MODERAT (KATEGORI SEDANG)
print("🧪 SKENARIO KEPARAHAN SEDANG: Pacaran 2 Tahun, Putus 2 Bulan Lalu, Masih Transisi Emosional\n")

res_sedang = predict_heartbreak_severity(
    nama='Budi Pratama',
    umur=23,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='bulan',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Kadang-kadang',
    frekuensi_medsos='Kadang-kadang'
)


In [ ]:
# CELL 4.3: SKENARIO KEPARAHAN RENDAH (KATEGORI RINGAN / RECOVERED)
print("🧪 SKENARIO KEPARAHAN RINGAN: Pacaran 2 Tahun, Sudah 2 Tahun Sejak Putus (Stabil & Move On)\n")

res_ringan = predict_heartbreak_severity(
    nama='Rian Ardiansyah',
    umur=25,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=2,
    sejak_putus_satuan='tahun',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Keputusan bersama',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Tidak pernah'
)


In [ ]:
# CELL 4.4: SKENARIO INPUT MINIMAL (HANYA WAJIB, OPSIONAL = NONE)
print("🧪 SKENARIO INPUT MINIMAL: Hanya Mengisi Umur, Lama Hubungan, & Sejak Putus\n")

res_minimal = predict_heartbreak_severity(
    nama='Siti Rahmawati',
    umur=21,
    lama_hubungan_nilai=1.5,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=5,
    sejak_putus_satuan='bulan',
    jenis_kelamin=None,
    pendidikan=None,
    siapa_mengakhiri=None,
    masih_komunikasi=None,
    frekuensi_medsos=None
)

print('\n🎉 SELURUH PENGUJIAN TINGKAT KEPARAHAN (RINGAN, SEDANG, BERAT) SELESAI & BERFUNGSI SEMPURNA!')


In [ ]:
# CELL 4.5: EKSPERIMEN PERBANDINGAN KHUSUS — UMUR 18 (SMA/SEKOLAH) VS UMUR 22 (S1/KERJA)
print("🧪 EKSPERIMEN PERBANDINGAN: Menguji Efek Perbedaan Pola Pikir Umur & Pendidikan pada Kondisi Putus yang Sama\n")

print("--- KASUS A: UMUR 18 TAHUN (MASIH SEKOLAH / SMA) ---")
res_a = predict_heartbreak_severity(
    nama='Siswa A (18 Tahun)',
    umur=18,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=1,
    sejak_putus_satuan='bulan',
    jenis_kelamin='Laki-laki',
    pendidikan='SMA/Sederajat',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Kadang-kadang',
    frekuensi_medsos='Kadang-kadang'
)

print("--- KASUS B: UMUR 22 TAHUN (SUDAH KULIAH/KERJA / S1) ---")
res_b = predict_heartbreak_severity(
    nama='Dewasa B (22 Tahun)',
    umur=22,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=1,
    sejak_putus_satuan='bulan',
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Kadang-kadang',
    frekuensi_medsos='Kadang-kadang'
)

# Ringkasan Tabel Perbandingan
comp_df = pd.DataFrame([
    {
        'Kasus': 'Kasus A: Remaja (Sekolah)',
        'Umur': 18,
        'Pendidikan': 'SMA',
        'Skor Distres (%)': f"{res_a['probabilitas_distres']:.1f}%",
        'Kategori': res_a['kategori_severity'],
        'Kerentanan Emosi': f"{res_a['info_durasi']['emotional_vulnerability_index']:.3f}"
    },
    {
        'Kasus': 'Kasus B: Dewasa Awal (Kerja/Kuliah)',
        'Umur': 22,
        'Pendidikan': 'S1',
        'Skor Distres (%)': f"{res_b['probabilitas_distres']:.1f}%",
        'Kategori': res_b['kategori_severity'],
        'Kerentanan Emosi': f"{res_b['info_durasi']['emotional_vulnerability_index']:.3f}"
    }
])

print("📊 TABEL RINGKASAN PERBANDINGAN DIFERENSIASI UMUR & PENDIDIKAN:")
display(comp_df)


In [ ]:
# CELL 4.6: PENGUJIAN TAHAP 6 BULAN SEJAK PUTUS (EFEK PEMULIHAN KOGNITIF DEWASA VS REMAJA)
print("🧪 PENGUJIAN 6 BULAN SEJAK PUTUS: Menguji Pemulihan Kognitif Umur 18 vs Umur 22\n")

print("--- KASUS A: UMUR 18 TAHUN (SMA), PUTUS SUDAH 6 BULAN ---")
res_6bln_18 = predict_heartbreak_severity(
    nama='Andi (18 Tahun)',
    umur=18,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=6,
    sejak_putus_satuan='bulan',       # <-- 6 BULAN SEJAK PUTUS
    jenis_kelamin='Laki-laki',
    pendidikan='SMA/Sederajat',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Jarang'
)

print("--- KASUS B: UMUR 22 TAHUN (S1), PUTUS SUDAH 6 BULAN ---")
res_6bln_22 = predict_heartbreak_severity(
    nama='Budi (22 Tahun)',
    umur=22,
    lama_hubungan_nilai=2,
    lama_hubungan_satuan='tahun',
    sejak_putus_nilai=6,
    sejak_putus_satuan='bulan',       # <-- 6 BULAN SEJAK PUTUS
    jenis_kelamin='Laki-laki',
    pendidikan='S1',
    siapa_mengakhiri='Pasangan yang mengakhiri',
    masih_komunikasi='Tidak sama sekali',
    frekuensi_medsos='Jarang'
)

# Ringkasan Tabel Perbandingan 6 Bulan
comp_6m_df = pd.DataFrame([
    {
        'Kasus': 'Andi (18 thn - SMA)',
        'Durasi Putus': '6 Bulan',
        'Skor Distres (%)': f"{res_6bln_18['probabilitas_distres']:.1f}%",
        'Tingkat Keparahan': res_6bln_18['kategori_severity'],
        'Status Pemulihan': 'Fase Transisi Adaptif'
    },
    {
        'Kasus': 'Budi (22 thn - S1)',
        'Durasi Putus': '6 Bulan',
        'Skor Distres (%)': f"{res_6bln_22['probabilitas_distres']:.1f}%",
        'Tingkat Keparahan': res_6bln_22['kategori_severity'],
        'Status Pemulihan': 'Pulih / Move On Lebih Cepat (Kematangan Kognitif)'
    }
])

print("📊 TABEL EVALUASI PEMULIHAN PADA 6 BULAN SEJAK PUTUS:")
display(comp_6m_df)
